In [2]:
import pdfplumber
import pandas as pd
import re
from pathlib import Path
from tqdm import tqdm
import sys
from datetime import datetime
parent_path = Path().resolve().parent  # notebook's parent directory
sys.path.append(str(parent_path))
from lib import util
import max.iaaf_points.score_calculator as iaaf

In [2]:
def standardize_name(name):
    if ',' in name:
        last, first = name.split(',', 1)
        return f"{first.strip()} {last.strip()}"
    return name.strip()

def standardize_birthyear(year):
    if pd.isna(year):
        return None
    
    year = str(year).strip()  # in String umwandeln
    
    # Sonderfall: '0' → 2000
    if year == '0':
        return 2000
    
    # Vierstelliges Jahr, z.B. '2014'
    if len(year) == 4:
        return int(year)
    
    # Zweistelliges Jahr, z.B. '80', '98', '00', '01'
    if len(year) == 2:
        yy = int(year)
        if yy >= 20:        # alles >=50 → 1900er
            return 1900 + yy
        else:               # alles <50 → 2000er
            return 2000 + yy
    
    # Einstellige Zahl, z.B. '9' → 2009?
    if len(year) == 1:
        return 2000 + int(year)
    
    # Alles andere: zurückgeben als int, falls möglich
    try:
        return int(year)
    except:
        return None

# Leistungsfehler (Leistung wird wg. Wind gekuerzt)

In [6]:
def fix_incomplete_leistung(df):
    # Maske: Leistung ist nur eine ganze Zahl (1–99) ohne Komma oder Doppelpunkt
    mask = df["leistung"].astype(str).str.fullmatch(r"\d{1,2}")

    # Kombinieren: leistung + wind (falls wind existiert)
    df.loc[mask & df["wind"].notna(), "leistung"] = (
        df.loc[mask, "leistung"].astype(str) + df.loc[mask, "wind"].astype(str)
    )

    # Wind auf None setzen
    df.loc[mask, "wind"] = None

    return df

In [3]:
def fix_manual_errors(df):
    """
    Korrigiert spezifische Tippfehler in der CSV, bevor die generelle Umwandlung stattfindet.
    Nutzt Boolesche Masken, um exakt nur die betroffenen Zeilen zu treffen.
    """
    
    # 1. Fall: Lena Posniak (02:07:44 -> 02:07,44)
    # Wir filtern auf Name UND den falschen Wert. Das verhindert, dass wir eine korrekte Zeit ändern.
    mask_lena = (
        (df['name'] == 'Lena Posniak') & 
        (df['leistung'] == '02:07:44') &
        (df['disziplin'] == '800 m') 
        # Optional: & (df['verein'] == 'LAC Erfurt') für maximale Sicherheit
    )
    # Nur wo die Maske wahr ist, ändern wir den Wert
    if mask_lena.any():
        df.loc[mask_lena, 'leistung'] = '02:07,44'
        print(f"Korrektur angewendet: {mask_lena.sum()} Zeile(n) für Lena Posniak korrigiert.")

    # 2. Fall: Ann-Kathrin Kopf (02:07:08 -> 02:07,08)
    mask_ann_kathrin = (
        (df['name'] == 'Ann-Kathrin Kopf') & 
        (df['leistung'] == '02:07:08') &
        (df['disziplin'] == '800 m')
    )
    if mask_ann_kathrin.any():
        df.loc[mask_ann_kathrin, 'leistung'] = '02:07,08'
        print(f"Korrektur angewendet: {mask_ann_kathrin.sum()} Zeile(n) für Ann-Kathrin Kopf korrigiert.")

    return df

In [4]:
def add_iaaf_scores(df):
    coeefs = iaaf.get_iaaf_coeffs()
    
    iaaf_scores = [None] * len(df)
    for i, row in df.iterrows():
        gender = 'men' if row['geschlecht'] == 'M' else 'women'
        discipline = iaaf.DISCIPLINE_TO_EVENT[row['disziplin']]
        mark = row['leistung']
        if row['disziplin'] in util.MEASUREMENT_TYPE['time']:
            mark = util.convert_time_to_seconds(mark)
        elif row['disziplin'] in util.MEASUREMENT_TYPE['meter']:
            mark = util.convert_meters_to_float(mark)
        elif row['disziplin'] in util.MEASUREMENT_TYPE['points']:
            mark = util.convert_points_to_int(mark)
        else:
            mark = None

        iaaf_score = iaaf.score_from_mark(gender, discipline, mark, coeefs)
        iaaf_scores[i] = iaaf_score
        
        

    df['iaaf_score'] = iaaf_scores
    return df


In [1]:
# add missing iaaf_scores
def add_missing_iaaf_scores(df):
    coeffs = iaaf.get_iaaf_coeffs()
    
    for i, row in df.iterrows():
        if pd.isna(row['iaaf_score']):
            gender = 'men' if row['geschlecht'] == 'M' else 'women'
            discipline = iaaf.DISCIPLINE_TO_EVENT[row['disziplin']]
            mark = row['leistung']
            if row['disziplin'] in util.MEASUREMENT_TYPE['time']:
                mark = util.convert_time_to_seconds(mark)
            elif row['disziplin'] in util.MEASUREMENT_TYPE['meter']:
                mark = util.convert_meters_to_float(mark)
            elif row['disziplin'] in util.MEASUREMENT_TYPE['points']:
                mark = util.convert_points_to_int(mark)
            else:
                mark = None
            
            iaaf_score = iaaf.score_from_mark(gender, discipline, mark, coeffs)
            df.at[i, 'iaaf_score'] = iaaf_score

    return df

In [3]:
df = pd.read_csv('final_Data_iaaf_scores_neu_v3.csv', sep=';')
df = add_missing_iaaf_scores(df)
df.to_csv('final_Data_iaaf_scores_neu_v3.csv', sep=';', index=False)

In [7]:


# Folder with PDFs
# pdf_folder = Path("data01-17")
# pdf_folder = Path("data24")
# pdf_folder = Path("data18-22")

# Regex for result lines (Platz wird ignoriert)
# 01-17
DEBUG = False

# 01-17
line_pattern_01_17 = re.compile(
    r"^(?:([1-9]\d?|50)\s+)?"                        # optional Platz
    r"(?P<leistung>("
    r"\d{1,2},\d{1,2}|"                              # SS,S oder SS,SS
    r"\d{1,2}:\d{2},\d{2}|"                          # M:SS,SS
    r"\d{1,2}:\d{2}:\d{2}|"                          # H:MM:SS
    r"\d{1,2}:\d{2}|"                                # M:SS
    r"\d{1,5}(?:\.\d{3})?"                            # Punkte, z.B. 8307
    r"))\s*"                                         # Leistung
    # Wind: erlaubt (±x,x) oder ±x,x mit/ohne Leerzeichen
    r"(?:(?:\(?\s*(?P<wind>[+-]\d,\d)\s*\)?)\s+)?"
    r"(?P<name>[A-Za-zÄÖÜäöüß ,\-]+?)\s+"            # Name
    r"(?P<geburtsjahr>\d{2,4})\s+"                   # Geburtsjahr
    r"(?P<verein>[A-Za-zÄÖÜäöüß0-9 .\-()/]+)\s+"     # Verein
    r"(?P<datum>(?:\d{1,2}\./\d{1,2}\.\d{2}\.)|\d{2}\.\d{2}\.(?:\d{2,4})?)"             # Datum
    r"(?P<ort>[A-Za-zÄÖÜäöüß .\-()/]+)$"             # Ort
)


# 24
line_pattern_24 = re.compile(
    r"^(?:[1-9]\d?|50)\s+"                              # Platz 1–50, zwingend
    r"(?P<leistung>("
    r"\d{1,2},\d{1,2}|"                                 # SS,S oder SS,SS
    r"\d{1,2}:\d{2},\d{2}|"                              # M:SS,SS
    r"\d{1,2}:\d{2}:\d{2}|"                             # H:MM:SS
    r"\d{1,2}:\d{2}|"                                   # M:SS
    r"\d{1,5}(?:\.\d{3})?"                              # Punkte, z.B. 8307
    r"))\s*"  
    r"(?:(?P<wind>[+-]?\d,\d)\s+)?"                    # optionaler Wind
    r"(?P<name>[A-Za-zÄÖÜäöüß'´`\- ]+?)\s+"            # Name
    r"(?P<geburtsjahr>\d{4})"                          # Geburtsjahr
    r"(?P<verein>[A-Za-zÄÖÜäöüß0-9 .\-()/]+?)\s+"      # Verein, direkt danach möglich
    r"(?P<datum>\d{2}\.\d{2}\.\d{4})\s+"               # Datum
    r"(?P<ort>[A-Za-zÄÖÜäöüß .\-()]+)$"                # Ort
)

# 23
line_pattern_23 = re.compile(
    r"^(?:([1-9]\d?|50)\s+)?"                        # Platz optional (1–50)
    r"(?P<leistung>("
    r"\d{1,2},\d{1,2}|"                              # SS,S oder SS,SS
    r"\d{1,2}:\d{2},\d{2}|"                          # M:SS,SS
    r"\d{1,2}:\d{2}:\d{2}|"                          # H:MM:SS
    r"\d{1,2}:\d{2}|"                                # M:SS
    r"\d{1,5}(?:\.\d{3})?"                               # Punkte, z.B. 8307
    r"))\s*"  
    r"(?:(?P<wind>\(?[+-]?\d,\d\)?)\s+)?"           # optionaler Wind, auch in Klammern
    r"(?P<name>[A-Za-zÄÖÜäöüß'´`\- ]+?)\s+"         # Name
    r"(?P<geburtsjahr>\d{4})\s+"                    # Geburtsjahr
    r"(?P<verein>[A-Za-zÄÖÜäöüß0-9 .\-()/]+?)\s+"   # Verein
    r"(?P<datum>\d{2}\.\d{2}\.\d{4})\s+"            # Datum
    r"(?P<ort>[A-Za-zÄÖÜäöüß .\-()/]+)$"            # Ort
)

# 18-22
line_pattern_18_22 = re.compile(
    r"^(?:([1-9]\d?|50)\s+)?"                              # Platz optional
    r"(?P<leistung>("
        r"\d{1,2},\d{2}|"                                 # SS,SS
        r"\d{1,2}:\d{2},\d{2}|"                           # M:SS,SS
        r"\d{1,2}:\d{2}:\d{2}|"                           # H:MM:SS
        r"\d{1,2}:\d{2}|"                                 # M:SS
        r"\d{1,5}(?:\.\d{3})?"                            # Punkte, z.B. 8307
    r"))\s*"
    r"(?:(?:\(?\s*(?P<wind>[+-]?\d,[0-9])\s*\)?)\s+)?"     # Wind optional, auch mit ()
    r"(?P<name>[A-Za-zÄÖÜäöüß'´`\- ]+?)\s+"                # Name
    r"(?P<geburtsjahr>\d{2,4})\s+"                         # Geburtsjahr
    r"(?P<verein>[A-Za-zÄÖÜäöüß0-9 .\-()/]+?)\s+"          # Verein
    r"(?P<datum>\d{2}\.\d{2}(?:\.\d{2,4})?)\.?\s+"         # HIER IST DIE KORREKTUR: \.? hinzugefügt
    r"(?P<ort>[A-Za-zÄÖÜäöüß .\-()/]+)$",                  # Ort
    re.UNICODE
)

pdf_year = {
    # "data01-17": line_pattern_01_17,
    # "data18-22": line_pattern_18_22,
    # "data24": line_pattern_24,
    # "data23": line_pattern_23,
    "data25": line_pattern_24
}

# Laufdisziplinen priorisiert (lange zuerst)
# lauf_pattern = (
#     r"10\s?km Straßengehen|20\s?km Straßengehen|50\s?km Straßengehen|"
#     r"(?:10|20|50)\s?km\s?Gehen|(?:10|20|50)\s?k\s?Gehen|"
#     r"(?:5|10|20|50|100)\s?km|"
#     r"(?:1\.?000|1\.?500|2\.?000|3\.?000|5\.?000|10\.?000)\s?m\s?(?:Bahngehen|Gehen|Hindernis|Hürden)?|"
#     r"(?:60|80|100|110|200|300|400|800|1000|1500|2000|3000|5000|10000)\s?m\s?(?:Hindernis|Hürden)?|"
#     r"^(60|80|100|110|200|300|400|800|1000|1500|2000|3000|5000|10000)\s*[\u00A0\u202F]?\s*m\b"
#     r"Halbmarathon|Marathon"
# )

lauf_pattern = (
    r"("
    r"10\s?km Straßengehen|20\s?km Straßengehen|50\s?km Straßengehen|"
    r"(?:10|20|50)\s?km\s?Gehen|(?:10|20|50)\s?k\s?Gehen|"
    r"(?:5|10|20|50|100)\s?km|"
    r"(?:1[ .]?000|1[ .]?500|2[ .]?000|3[ .]?000|5[ .]?000|10[ .]?000)\s?m\s?(?:Bahngehen|Gehen|Hindernis|Hürden)?|"
    r"(?:60|80|100|110|200|300|400|800|1000|1500|2000|3000|5000|10000)\s?m\s?(?:Hindernis|Hürden)?|"
    r"^(60|80|100|110|200|300|400|800|1000|1500|2000|3000|5000|10000)\s*[\u00A0\u202F]?\s*m\b|"
    r"Halbmarathon|Marathon"
    r")"
)

# lauf_regex = re.compile(lauf_pattern)


# Komplette Disziplin-Regex
discipline_pattern = re.compile(
    r"^("
    + lauf_pattern + "|" +
    r"Weitsprung|Hochsprung|Dreisprung|Stabhochsprung|"
    r"Kugelstoß|Speerwurf|Diskuswurf|Hammerwurf|"
    r"Zehnkampf|Siebenkampf|Fünfkampf|10-Kampf|7-Kampf|5-Kampf"
    r")"
)

discipline_standardization = {
    # Läufe
    "60m": "60 m", "100m": "100 m", "200m": "200 m", "300m": "300 m",
    "400m": "400 m", "800m": "800 m", 
    "1000m": "1000 m", "1.000 m": "1000 m",
    "1500m": "1500 m", "1.500 m": "1500 m",
    "2000m": "2000 m", "2.000 m": "2000 m",
    "3000m": "3000 m", "3.000 m": "3000 m",
    "5000m": "5000 m", "5.000 m": "5000 m",
    "10000m": "10 000 m", "10.000 m": "10 000 m", "10000 m": "10 000 m",
    "5km": "5 km", "10km": "10 km", "20km": "20 km",
    "10 km Straßengehen": "10 km Gehen",
    "10 km Gehen": "10 km Gehen",
    "20 km Gehen": "20 km Gehen",
    "20 km Straßengehen": "20 km Gehen",
    "3000 m Bahngehen": "3000 m Gehen", "3.000 m Bahngehen": "3000 m Gehen",
    "5000 m Bahngehen": "5000 m Gehen", "5.000 m Bahngehen": "5000 m Gehen",
    "10 000 m Bahngehen": "10 000 m Gehen", "10000 m Bahngehen": "10 000 m Gehen", "10.000 m Bahngehen": "10 000 m Gehen",

    # Hürden
    "100 m Hürden": "100 m Huerden", "110 m Hürden": "110 m Huerden", "400 m Hürden": "400 m Huerden",

    #Hindernis
    "1500 m Hindernis": "1500 m Hindernis", "1.500 m Hindernis": "1500 m Hindernis",
    "2000 m Hindernis": "2000 m Hindernis", "2.000 m Hindernis": "2000 m Hindernis",
    "3000 m Hindernis": "3000 m Hindernis", "3.000 m Hindernis": "3000 m Hindernis",

    # Sprung / Wurf
    "Weitsprung": "Weitsprung", "Hochsprung": "Hochsprung", "Dreisprung": "Dreisprung",
    "Stabhochsprung": "Stabhochsprung", "Kugelstoß": "Kugelstoss", "Speerwurf": "Speerwurf",
    "Diskuswurf": "Diskuswurf", "Hammerwurf": "Hammerwurf",

    # Mehrkampf
    "Zehnkampf": "Zehnkampf", "10-Kampf": "Zehnkampf", "Siebenkampf": "Siebenkampf", "7-Kampf": "Siebenkampf",
}


# Funktion zum Parsen von Dateinamen
def parse_filename(filename):
    stem = Path(filename).stem

    # Jahr
    year_match = re.search(r"bestenliste(\d{4})", stem)
    year = year_match.group(1) if year_match else ""

    # Geschlecht
    if any(s in stem for s in ["maennliche", "maenner", "junioren"]):
        gender = "M"
    elif any(s in stem for s in ["weibliche", "frauen", "juniorinnen"]):
        gender = "W"
    else:
        gender = ""

    # Altersklasse
    if "maenner" in stem:
        age_class = "Männer"
    elif "frauen" in stem:
        age_class = "Frauen"
    elif "junioren" in stem or "juniorinnen" in stem:
        age_class = "U23"
    else:
        u_match = re.search(r"U\d{2}$", stem)
        if u_match:
            age_class = u_match.group(0)
        else:
            num_match = re.search(r"(\d{1,2})$", stem)
            age_class = num_match.group(1) if num_match else ""

    return year, gender, age_class

# Funktion zum Ersetzen von Umlauten
def replace_umlauts(text):
    if not isinstance(text, str):
        return text
    replacements = {
        "ä": "ae", "ö": "oe", "ü": "ue",
        "Ä": "Ae", "Ö": "Oe", "Ü": "Ue",
        "ß": "ss"
    }
    for orig, repl in replacements.items():
        text = text.replace(orig, repl)
    return text

# Parse PDF
def parse_pdf(pdf_path):
    
    year, gender, age_class = parse_filename(pdf_path.name)

    results = []
    current_discipline = None
    ignore_section = False
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue
            for line in text.split("\n"):
                line = line.strip()
                if not line:
                    continue
                if line.startswith("Ausländer"):
                    ignore_section = True
                    continue
                if re.search(r"(?i)(mannschaft|staffel|\d+\s?x\s?\d+\s?m)", line):
                    ignore_section = True
                    continue
                
                # Wenn wir im Ausländer-Abschnitt sind, bis zur nächsten Disziplin ignorieren
                if ignore_section:
                    # Disziplin erkannt → Ausländer-Abschnitt endet
                    if discipline_pattern.match(line):
                        ignore_section = False
                    else:
                        continue
                # Skip team results
                if "Mannschaft" in line or "Mannschaftswertung" in line:
                    continue
                # Skip lines in parentheses (individuals within team)
                if line.startswith("(") and line.endswith(")"):
                    continue
                # Skip Staffel lines
                if re.search(r"\dx\d+", line):
                    ignore_section = True
                    current_discipline = None
                    continue
                # Discipline detected
                match_d = discipline_pattern.match(line)
                if match_d:
                    current_discipline = match_d.group(0).strip()
                    # Standardisierte Schreibweise (immer Leerzeichen)
                    current_discipline = discipline_standardization.get(current_discipline, current_discipline)
                    if DEBUG:
                        if (current_discipline == "Siebenkampf" or current_discipline == "Zehnkampf"):
                            print("Debug")
                    continue
                # Result line detected
                match = line_pattern.match(line)

                if match and current_discipline:
                    measurement_key = util.get_measurement_key(current_discipline)
                    if not measurement_key:
                        continue
                    data = match.groupdict()

                    performance = data['leistung']
                    allowed_perfomance_patterns = util.PERFORMANCE_PATTERNS[measurement_key]
                    match_perf = False
                    for pattern in allowed_perfomance_patterns:
                        if re.match(pattern, performance):
                            match_perf = True
                            break
                    if not match_perf:
                        continue
                    data['name'] = standardize_name(data['name'])
                    data['geburtsjahr'] = standardize_birthyear(data['geburtsjahr'])
                    data.update({
                        "jahr": year,
                        "geschlecht": gender,
                        "altersklasse": age_class,
                        "disziplin": current_discipline
                    })
                    results.append(data)
    return results


# Alle PDFs durchgehen

all_results = []

if DEBUG:
    idx = 0
    pdf_folder = Path("../Data_pdf/data18-22")
    line_pattern = line_pattern_18_22
    for pdf_file in pdf_folder.glob("*.pdf"):
        if "2022" not in pdf_file.name or "maenner" not in pdf_file.name:
            continue
        print(f"Processing: {pdf_file.name}")
        all_results.extend(parse_pdf(pdf_file))
        idx += 1
        if idx >= 10:
            break
else:
    for key in pdf_year:
        pdf_folder = Path(f"../Data_pdf/{key}")
        line_pattern = pdf_year[key]
        pdf_files = list(pdf_folder.glob("*.pdf"))
        
        print(f"\n📂 Verarbeite Ordner: {pdf_folder} ({len(pdf_files)} Dateien)")
        
        for pdf_file in tqdm(pdf_files, desc=f"{key}", unit="pdf"):
            all_results.extend(parse_pdf(pdf_file))


# DataFrame
df = pd.DataFrame(all_results)

# Umlaut-/Sonderzeichen konvertieren
df = df.applymap(replace_umlauts)

print(df)

# Spalten sortieren
df = df[["jahr", "geschlecht", "altersklasse", "disziplin",
         "leistung", "wind", "name", "geburtsjahr",
         "verein", "datum", "ort"]]


# CSV speichern
if not DEBUG:
    df = fix_incomplete_leistung(df)
    df = fix_manual_errors(df)
    df = add_iaaf_scores(df)
    date_str = datetime.now().strftime("%Y%m%d_%H%M")
    # df.to_csv(f"Data_{date_str}.csv", index=False, sep=";")
    df.to_csv("Data2025.csv", index=False, sep=";")

print(f"✅ Fertig! Daten gespeichert in: Data_{date_str}.csv")
# df.head(10)
df



📂 Verarbeite Ordner: ../Data_pdf/data25 (12 Dateien)


data25: 100%|██████████| 12/12 [00:23<00:00,  1.99s/pdf]
/var/folders/lf/d2nc392142b837mf8sypzlx00000gn/T/ipykernel_6225/1478686294.py:326: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(replace_umlauts)


      leistung  wind                name  geburtsjahr                verein  \
0        10,31  +1,4      Jakob Kemminer         2007     LAC Quelle Fuerth   
1        10,37  +1,5        Ben Troester         2007        TSG Lennestadt   
2        10,46  +1,5       Mika Dedekind         2007               LC Jena   
3        10,51  +1,5       Felix Schulze         2006          Hamburger SV   
4        10,51  +1,4       Alvin Mawumba         2007    TV Wattenscheid 01   
...        ...   ...                 ...          ...                   ...   
11388     6530  None   Christoph Ewinger         1991      VfL Sindelfingen   
11389     6523  None         Finn Schulz         2003  LG Steinlach-Zollern   
11390     6504  None       Till Ruediger         2003             LG Aachen   
11391     6486  None  Valentin Schneider         1998          TSV Rottweil   
11392     6483  None      Lukas Kussmann         1997              SV Halle   

            datum                     ort  jahr ges

,jahr,geschlecht,altersklasse,disziplin,leistung,wind,name,geburtsjahr,verein,datum,ort,iaaf_score
0,2025,M,20,100 m,"10,31","+1,4",Jakob Kemminer,2007,LAC Quelle Fuerth,28.06.2025,Mannheim,1102
1,2025,M,20,100 m,"10,37","+1,5",Ben Troester,2007,TSG Lennestadt,14.06.2025,Regensburg,1082
2,2025,M,20,100 m,"10,46","+1,5",Mika Dedekind,2007,LC Jena,07.06.2025,Berlin-Lichterfelde,1053
3,2025,M,20,100 m,"10,51","+1,5",Felix Schulze,2006,Hamburger SV,14.06.2025,Regensburg,1037
4,2025,M,20,100 m,"10,51","+1,4",Alvin Mawumba,2007,TV Wattenscheid 01,28.06.2025,Mannheim,1037
...,...,...,...,...,...,...,...,...,...,...,...,...
11388,2025,M,Maenner,Zehnkampf,6530,None,Christoph Ewinger,1991,VfL Sindelfingen,21.06.2025,Filderstadt,901
11389,2025,M,Maenner,Zehnkampf,6523,None,Finn Schulz,2003,LG Steinlach-Zollern,21.06.2025,Filderstadt,900
11390,2025,M,Maenner,Zehnkampf,6504,None,Till Ruediger,2003,LG Aachen,21.06.2025,Filderstadt,897
11391,2025,M,Maenner,Zehnkampf,6486,None,Valentin Schneider,1998,TSV Rottweil,21.06.2025,Filderstadt,895


In [11]:
df = pd.read_csv("Data.csv", sep=";")

discipline_standardization = {
    # Läufe
    "60m": "60 m", "100m": "100 m", "200m": "200 m", "300m": "300 m",
    "400m": "400 m", "800m": "800 m", 
    "1000m": "1000 m", "1.000 m": "1000 m",
    "1500m": "1500 m", "1.500 m": "1500 m",
    "2000m": "2000 m", "2.000 m": "2000 m",
    "3000m": "3000 m", "3.000 m": "3000 m",
    "5000m": "5000 m", "5.000 m": "5000 m",
    "10000m": "10 000 m", "10.000 m": "10 000 m", "10000 m": "10 000 m",
    "5km": "5 km", "10km": "10 km", "20km": "20 km",
    "10 km Straßengehen": "10 km Gehen",
    "10 km Gehen": "10 km Gehen",
    "20 km Gehen": "20 km Gehen",
    "20 km Straßengehen": "20 km Gehen",
    "3000 m Bahngehen": "3000 m Gehen", "3.000 m Bahngehen": "3000 m Gehen",
    "5000 m Bahngehen": "5000 m Gehen", "5.000 m Bahngehen": "5000 m Gehen",
    "10 000 m Bahngehen": "10 000 m Gehen", "10000 m Bahngehen": "10 000 m Gehen", "10.000 m Bahngehen": "10 000 m Gehen", "10000 m Gehen": "10 000 m Gehen",

    # Hürden
    "100 m Hürden": "100 m Huerden", "110 m Hürden": "110 m Huerden", "400 m Hürden": "400 m Huerden",

    #Hindernis
    "1500 m Hindernis": "1500 m Hindernis", "1.500 m Hindernis": "1500 m Hindernis",
    "2000 m Hindernis": "2000 m Hindernis", "2.000 m Hindernis": "2000 m Hindernis",
    "3000 m Hindernis": "3000 m Hindernis", "3.000 m Hindernis": "3000 m Hindernis",

    # Sprung / Wurf
    "Weitsprung": "Weitsprung", "Hochsprung": "Hochsprung", "Dreisprung": "Dreisprung",
    "Stabhochsprung": "Stabhochsprung", "Kugelstoß": "Kugelstoss", "Speerwurf": "Speerwurf",
    "Diskuswurf": "Diskuswurf", "Hammerwurf": "Hammerwurf",

    # Mehrkampf
    "Zehnkampf": "Zehnkampf", "Siebenkampf": "Siebenkampf"
}

df["disziplin"] = df["disziplin"].map(discipline_standardization).fillna(df["disziplin"])

print(df["disziplin"].unique())

['100 m' '200 m' '400 m' '800 m' '1500 m' '3000 m' '5000 m' '10 km'
 'Halbmarathon' 'Marathon' '100 km' '100 m Huerden' '400 m Huerden'
 '3000 m Hindernis' 'Hochsprung' 'Stabhochsprung' 'Weitsprung'
 'Dreisprung' 'Kugelstoss' 'Diskuswurf' 'Hammerwurf' 'Speerwurf'
 '5000 m Gehen' '10 km Gehen' '20 km Gehen' '110 m Huerden'
 '2000 m Hindernis' '300 m' '1000 m' '5 km' '3000 m Gehen' '50 km Gehen'
 '2000 m' '10 000 m Gehen' '1500 m Hindernis' '10 000 m' 'Siebenkampf'
 'Zehnkampf']


In [12]:
df.to_csv("Data.csv", index=False, sep=";")

## Append Dataframe

In [8]:
old_df = pd.read_csv("final_Data_iaaf_scores_neu.csv", sep=";")
new_df = pd.read_csv("Data2025.csv", sep=";")

merged_df = pd.concat([old_df, new_df], ignore_index=True)
merged_df.to_csv("final_Data_iaaf_scores_neu_v2.csv", index=False, sep=";")